# 03. ML 전달 데이터 확인
최종 데이터 상태와 universe 정합성을 한 번에 확인합니다.

In [1]:
from pathlib import Path
import sys
import pandas as pd
PROJECT_ROOT = Path('/Users/choedasom/lab_middle_project')
assert (PROJECT_ROOT / 'src' / 'validate.py').is_file(), f'경로 확인 필요: {PROJECT_ROOT}'
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

for m in list(sys.modules):
    if m == 'src' or m.startswith('src.'):
        del sys.modules[m]

from src.validate import validate_ml_dataset
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
final_df = pd.read_parquet(PROCESSED_DIR / 'final_df.parquet')
coverage_df = pd.read_csv(PROCESSED_DIR / 'coverage_df.csv', parse_dates=['actual_start_date', 'actual_end_date'])
final_missing_df = pd.read_csv(PROCESSED_DIR / 'final_missing_df.csv')
sp500_universe = pd.read_csv(RAW_DIR / 'sp500_universe.csv')

In [2]:
summary = {
 'final_df shape': final_df.shape, '최종 종목 수': final_df['Ticker'].nunique(),
 '전체 행 수': len(final_df), '날짜 범위': (final_df['Date'].min(), final_df['Date'].max()),
 '컬럼 목록': final_df.columns.tolist(), 'final_missing_df 종목 수': final_missing_df['Ticker'].nunique(),
 'short_history 종목 수': int(coverage_df['short_history'].sum()),
 'Ticker/Date 중복': int(final_df.duplicated(['Ticker','Date']).sum()),
 '필수 컬럼 결측': final_df[['Ticker','Date','Open','High','Low','Close','Volume']].isna().sum().to_dict()}

In [3]:
for key, value in summary.items(): print(f'{key}: {value}')
display(coverage_df.describe(include='all').T)
display(final_missing_df.groupby('fail_stage').size().rename('count'))

final_df shape: (1661688, 8)
최종 종목 수: 707
전체 행 수: 1661688
날짜 범위: (Timestamp('2016-01-04 00:00:00'), Timestamp('2026-06-30 00:00:00'))
컬럼 목록: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'source']
final_missing_df 종목 수: 22
short_history 종목 수: 130
Ticker/Date 중복: 0
필수 컬럼 결측: {'Ticker': 0, 'Date': 0, 'Open': 0, 'High': 0, 'Low': 0, 'Close': 0, 'Volume': 0}


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Ticker,707,707,A,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_rows,707.0,NaN,NaN,NaN,2350.336634,24.0,2637.0,2637.0,2637.0,2637.0,665.976318
actual_start_date,707,NaN,NaN,NaN,2016-06-24 13:08:13.917963008,2016-01-04 00:00:00,2016-01-04 00:00:00,2016-01-04 00:00:00,2016-01-04 00:00:00,2026-05-27 00:00:00,NaN
actual_end_date,707,NaN,NaN,NaN,2025-11-01 15:40:59.405940736,2016-02-08 00:00:00,2026-06-30 00:00:00,2026-06-30 00:00:00,2026-06-30 00:00:00,2026-06-30 00:00:00,NaN
n_price_imputed,707.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
n_ohlc_inconsistent,707.0,NaN,NaN,NaN,0.500707,0.0,0.0,0.0,0.0,78.0,3.463846
n_volume_missing,707.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sources,707,2,yahoo,604,NaN,NaN,NaN,NaN,NaN,NaN,NaN
expected_rows_10y,707.0,NaN,NaN,NaN,2738.0,2738.0,2738.0,2738.0,2738.0,2738.0,0.0
coverage_10y,707.0,NaN,NaN,NaN,0.858405,0.0088,0.9631,0.9631,0.9631,0.9631,0.243231


fail_stage
collection    22
Name: count, dtype: int64

In [4]:
report = validate_ml_dataset(final_df, final_missing_df, sp500_universe, coverage_df)
print('[통과] 필수 검증 통과')
report

[통과] 필수 검증 통과


{'valid': True,
 'errors': [],
 'warnings': ['short_history 종목 130개(제외하지 않음)'],
 'final_shape': (1661688, 8),
 'n_final_tickers': 707,
 'n_missing_tickers': 22,
 'n_universe_tickers': 729}